# 03 — Analysis & Visualization

**Project:** Post-Only (A) vs Trajectory (B) Supervision for Continual Tool-Use Learning

In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D

## 1. Load Results

In [ ]:
result_files = [
    'results_A_seed42.json',
    'results_B_seed42.json',
    'results_A+_seed42.json',
]

results = {}
for fname in result_files:
    if os.path.exists(fname):
        with open(fname) as f:
            r = json.load(f)
        results[r['condition']] = r
        print(f"Loaded: {fname} (condition {r['condition']})")
    else:
        print(f"NOT FOUND: {fname}")

print(f"\nConditions loaded: {list(results.keys())}")

## 2. Continual Learning Metrics

In [ ]:
PRIMARY_MATRIX_KEY = 'eval_full_acc'
PRIMARY_ZS_KEY = 'full_acc'
SECONDARY_MATRIX_KEY = 'eval_acc'
SECONDARY_ZS_KEY = 'name_acc'

def compute_cl_metrics(result, matrix_key=PRIMARY_MATRIX_KEY, zero_shot_key=PRIMARY_ZS_KEY):
    acc = np.array(result[matrix_key])
    n = acc.shape[0]

    aa = np.mean(acc[-1, :])
    bwt = np.mean([acc[-1, j] - acc[j, j] for j in range(n - 1)]) if n > 1 else 0.0

    zs = result.get('zero_shot', {}).get(zero_shot_key, [])
    fwt_terms = [acc[j - 1, j] - zs[j] for j in range(1, n) if len(zs) > j]
    fwt = np.mean(fwt_terms) if fwt_terms else 0.0

    forgetting = []
    for j in range(n - 1):
        best = max(acc[i, j] for i in range(j, n))
        forgetting.append(best - acc[-1, j])
    avg_forgetting = np.mean(forgetting) if forgetting else 0.0

    aulc_vals = []
    for j in range(n):
        curve = [acc[i, j] for i in range(j, n)]
        if len(curve) > 1:
            aulc = np.trapz(curve, dx=1.0) / (len(curve) - 1)
        else:
            aulc = curve[0]
        aulc_vals.append(aulc)
    avg_aulc = np.mean(aulc_vals)

    return {
        'Average Accuracy (AA)': aa,
        'Backward Transfer (BWT)': bwt,
        'Forward Transfer (FWT)': fwt,
        'Average Forgetting': avg_forgetting,
        'Average AULC': avg_aulc,
    }

all_metrics = {}
name_metrics = {}
for cond, r in results.items():
    primary = compute_cl_metrics(r, PRIMARY_MATRIX_KEY, PRIMARY_ZS_KEY)
    secondary = compute_cl_metrics(r, SECONDARY_MATRIX_KEY, SECONDARY_ZS_KEY)
    all_metrics[cond] = primary
    name_metrics[cond] = secondary

    print(f"\nCondition {cond} (primary = exact full-call):")
    for k, v in primary.items():
        print(f"  {k}: {v:.4f}")

    print(f"Condition {cond} (secondary = API name):")
    for k, v in secondary.items():
        print(f"  {k}: {v:.4f}")


## 3. Summary Table

In [ ]:
conds = sorted(results.keys())
metric_keys = [
    'Average Accuracy (AA)',
    'Backward Transfer (BWT)',
    'Forward Transfer (FWT)',
    'Average Forgetting',
    'Average AULC',
]

print("=" * 80)
print("CONTINUAL LEARNING METRICS")
print("=" * 80)
header = f"{'Metric':<28}" + "".join(f"{c:>16}" for c in conds)
print(header)
print("-" * (28 + 16 * len(conds)))
for key in metric_keys:
    row = f"{key:<28}"
    for c in conds:
        row += f"{all_metrics[c][key]:>16.4f}"
    print(row)

In [ ]:
# Full evaluation matrices
for cond in conds:
    r = results[cond]
    n = len(r[PRIMARY_MATRIX_KEY])
    print(f"\n{'=' * 72}")
    print(f"EVAL MATRIX — Condition {cond} (Exact Full-Call Accuracy)")
    print(f"{'=' * 72}")
    print(f"{'':>14}", end="")
    for j in range(n):
        print(f"{'D' + str(j+1):>10}", end="")
    print()
    for i in range(n):
        print(f"After D{i+1}:    ", end="")
        for j in range(n):
            print(f"{r[PRIMARY_MATRIX_KEY][i][j]:>10.1%}", end="")
        print()


## 4. Visualizations

In [ ]:
sns.set_style("whitegrid")
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle(
    "Post-Only (A) vs Trajectory (B) vs Token-Matched (A+)\n"
    "Continual Tool-Use Learning — Mistral-7B-Instruct-v0.3, QLoRA, API-Bank",
    fontsize=13, fontweight='bold',
)

n = len(list(results.values())[0][PRIMARY_MATRIX_KEY])
blabels = [f'D{i+1}' for i in range(n)]
colors = {'A': '#e74c3c', 'B': '#2ecc71', 'A+': '#3498db'}

# Plot 1 & 2: Loss Heatmaps
for idx, cond in enumerate(['A', 'B']):
    if cond not in results:
        continue
    ax = axes[0, idx]
    data = np.array(results[cond]['eval_loss'])
    im = ax.imshow(data, cmap='YlOrRd', aspect='auto')
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(blabels)
    ax.set_yticklabels([f'After {l}' for l in blabels])
    ax.set_xlabel("Evaluated on"); ax.set_ylabel("Training stage")
    ax.set_title(f"Eval Loss — {cond}")
    for i in range(n):
        for j in range(n):
            c = 'white' if data[i,j] > np.median(data) else 'black'
            ax.text(j, i, f'{data[i,j]:.2f}', ha='center', va='center',
                    fontsize=6, color=c)
    plt.colorbar(im, ax=ax, shrink=0.8)

# Plot 3: Forgetting Curves
ax = axes[0, 2]
for cond in ['A', 'B']:
    if cond not in results:
        continue
    acc = np.array(results[cond][PRIMARY_MATRIX_KEY])
    for j in range(n):
        stages = list(range(j, n))
        vals = [acc[i, j] for i in stages]
        style = 'o--' if cond == 'A' else 's-'
        alpha = 0.5 if cond == 'A' else 0.9
        ax.plot(stages, vals, style, color=f'C{j}', alpha=alpha)
ax.set_xticks(range(n))
ax.set_xticklabels([f'After {l}' for l in blabels])
ax.set_ylabel("Exact Full-Call Accuracy"); ax.set_title("Forgetting Curves")
legend_lines = [
    Line2D([0],[0], linestyle='--', marker='o', color='gray'),
    Line2D([0],[0], linestyle='-', marker='s', color='gray'),
]
ax.legend(legend_lines, ['A: Post-Only', 'B: Trajectory'])

# Plot 4: Final Accuracy
ax = axes[1, 0]
x = np.arange(n)
w = 0.8 / len(conds)
for ci, cond in enumerate(conds):
    vals = np.array(results[cond][PRIMARY_MATRIX_KEY])[-1]
    ax.bar(x + ci*w - 0.4 + w/2, vals, w,
           label=cond, color=colors.get(cond, 'gray'), alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(blabels)
ax.set_ylabel("Accuracy"); ax.set_title("Final Exact Full-Call Accuracy")
ax.legend(fontsize=9)

# Plot 5: Perplexity
ax = axes[1, 1]
for cond in ['A', 'B']:
    if cond not in results:
        continue
    ppl = np.array(results[cond]['eval_ppl'])
    for j in range(n):
        style = 'o--' if cond == 'A' else 's-'
        alpha = 0.5 if cond == 'A' else 0.9
        ax.plot(range(n), [ppl[i,j] for i in range(n)],
                style, color=f'C{j}', alpha=alpha)
ax.set_xticks(range(n))
ax.set_xticklabels([f'After {l}' for l in blabels])
ax.set_ylabel("Perplexity"); ax.set_title("Perplexity Over Time")
ax.legend(legend_lines, ['A: Post-Only', 'B: Trajectory'])

# Plot 6: CL Metrics
ax = axes[1, 2]
short_names = ['AA', 'BWT', 'FWT', 'Forget', 'AULC']
x = np.arange(len(metric_keys))
w = 0.8 / len(conds)
for ci, cond in enumerate(conds):
    vals = [all_metrics[cond][k] for k in metric_keys]
    ax.bar(x + ci*w - 0.4 + w/2, vals, w,
           label=cond, color=colors.get(cond, 'gray'), alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(short_names)
ax.set_ylabel("Score"); ax.set_title("CL Metrics Comparison (Full-Call)")
ax.legend(); ax.axhline(y=0, color='black', linewidth=0.5)

plt.tight_layout()
plt.savefig('final_results.png', dpi=150, bbox_inches='tight')
plt.savefig('final_results.pdf', bbox_inches='tight')
plt.show()
print("Saved: final_results.png, final_results.pdf")


## 5. Zero-Shot Baseline

In [ ]:
print("ZERO-SHOT BASELINE")
print("-" * 60)
for cond in conds:
    r = results[cond]
    zs = r.get('zero_shot', {})
    if not zs or not zs.get('name_acc'):
        continue
    print(f"\n  Condition {cond}:")
    for j in range(len(zs['name_acc'])):
        print(f"    D{j+1}: name={zs['name_acc'][j]:.1%}, "
              f"full={zs['full_acc'][j]:.1%}, "
              f"loss={zs['loss'][j]:.3f}")

## 6. Token Budget Analysis

In [ ]:
print("TOKEN BUDGET ANALYSIS")
print("=" * 60)
print("Primary CL metric source: exact full-call accuracy.")

if 'A' in all_metrics and 'B' in all_metrics:
    aa_a = all_metrics['A']['Average Accuracy (AA)']
    aa_b = all_metrics['B']['Average Accuracy (AA)']
    gap_raw = aa_b - aa_a
    print(f"\nA (post-only):  AA = {aa_a:.4f}")
    print(f"B (trajectory): AA = {aa_b:.4f}")
    print(f"Raw gap (B - A): {gap_raw:+.4f}")

if 'A+' in all_metrics and 'B' in all_metrics:
    aa_ap = all_metrics['A+']['Average Accuracy (AA)']
    gap_matched = aa_b - aa_ap
    print(f"\nA+ (token-matched): AA = {aa_ap:.4f}")
    print(f"Token-matched gap (B - A+): {gap_matched:+.4f}")

    if gap_raw > 0:
        explained = (1 - gap_matched / gap_raw) * 100
        print(f"\nToken budget explains {explained:.1f}% of the raw gap.")
        print(f"Remaining gap: {gap_matched:+.4f}")
        if gap_matched > 0:
            print("=> Trajectory benefit BEYOND just more tokens.")
        else:
            print("=> Gap fully explained by token budget.")

# Per-block comparison
if 'A' in results and 'B' in results:
    print(f"\nPer-block final exact full-call accuracy:")
    acc_a = np.array(results['A'][PRIMARY_MATRIX_KEY])[-1]
    acc_b = np.array(results['B'][PRIMARY_MATRIX_KEY])[-1]
    for j in range(len(acc_a)):
        diff = acc_b[j] - acc_a[j]
        print(f"  D{j+1}: A={acc_a[j]:.1%}, B={acc_b[j]:.1%}, diff={diff:+.1%}")


## 7. Export

In [ ]:
analysis = {
    'metrics': {c: m for c, m in all_metrics.items()},
    'conditions': list(results.keys()),
}
for cond, r in results.items():
    analysis[f'eval_acc_{cond}'] = r['eval_acc']
    analysis[f'zero_shot_{cond}'] = r.get('zero_shot', {})

with open('analysis_results.json', 'w') as f:
    json.dump(analysis, f, indent=2, default=str)

print("Saved: analysis_results.json")
print("\nFiles for paper:")
print("  final_results.png / .pdf")
print("  analysis_results.json")